# Controlled Generation- Place Exactly What You Want, Where You Want It

**Module 6 · Lesson 6.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Control Problem - Content vs Composition
- ControlNet - Condition on Structure
- IP-Adapter - An Image as a Prompt
- Composing Control - Multi-ControlNet and Regional Prompting
- Choosing and Tuning Control
- Control in Production

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

## The whole game: a prompt is content, a control is composition

> 🎬 **Analogy**
>
> **A film director works from a storyboard, not just a script.** The script (your prompt) says what happens - a robot, studio light. The storyboard (a ControlNet condition) fixes the exact framing and blocking: stand here, arm raised, camera at this angle. A mood board (an IP-Adapter reference) fixes the look. The actors and set - the base model and prompt from 6.2 - still do the work; the storyboard just constrains where everything goes.
> That is the whole lesson: **text gives you content, control gives you composition**. Everything here is a way to hand the model a blueprint - a structure map, a reference image, a per-region prompt - so you stop rolling seeds hoping the layout lands right.
> **Where the analogy breaks down:** a human director improvises around a storyboard. ControlNet follows it literally, in proportion to a "conditioning scale" dial (0 = loose, 1 = rigid) - so an over-tight storyboard (scale near 1) yields a stiff, traced-looking result. The model has no judgment to loosen the blueprint; you set that yourself.

This is why open **Stable Diffusion** matters for control work in a way closed tools like **Midjourney** and **DALL-E** do not: because the pipeline is open, you can bolt control models onto it. We build directly on Lesson 6.1 (the frozen denoiser that predicts noise - ControlNet attaches a branch to it) and Lesson 6.2 (the pipeline, the dials, img2img strength, LoRA, inpainting - all assumed here, not re-taught), running on **PyTorch** and the Hugging Face `diffusers` library. We stay scoped to the control layer. One forward hook to flag now: IP-Adapter works by encoding an image with CLIP, which is exactly what Lesson 6.4 opens up.

- **Explain** why text-to-image lacks spatial control, and how ControlNet adds it - a trainable branch that conditions the frozen denoiser on a structure map (edges, depth, pose).

- **Apply** the `diffusers` ControlNet and IP-Adapter pipelines - extract a condition with a preprocessor, set the conditioning scale, and use a reference image as a prompt.

- **Compare** the control methods (ControlNet vs IP-Adapter vs img2img vs a LoRA) and choose the right one for structure, style, subject, or region.

- **Analyze** a multi-control setup (multi-ControlNet, union models, regional prompting, control timing) and diagnose over- or under-conditioning.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against Lessons 6.1 and 6.2. Each one is load-bearing for today.

---

## 🎯 Concept 1: The Control Problem - Content vs Composition

### The Control Problem - Content vs Composition

A perfect prompt still can't fix the pose or layout. That gap is what every tool here closes.

**Ordering a cake by phone vs handing over a photo.** "A blue birthday cake with flowers" gets you a blue cake with flowers somewhere - but the layout is the baker's guess. Hand them a photo to match and the arrangement is fixed. Words pick the ingredients; the picture pins the design.

The gain: a prompt controls content; a control input pins composition. More adjectives will not fix layout - you need a blueprint.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> 🎯 The three axes of control
> - **Structure** - where things go: pose, layout, depth, edges. Tool: **ControlNet** (Step 2).
> - **Style / subject** - what it looks like, from a reference image. Tool: **IP-Adapter** (Step 3).
> - **Region** - different content in different areas of one image. Tool: **regional prompting** (Step 4).

#### 💡 What just happened

⚡ What just happened?
  A prompt is a strong lever over **content** and a weak one over **composition** - run it across seeds and the layout reshuffles every time. Controlled generation adds inputs that pin composition: a structure map, a reference image, or a per-region prompt. The rest of the lesson is the three tools that do it.

---

## 🎯 Concept 2: ControlNet - Condition on Structure

### ControlNet - Condition on Structure

Extract a structure map from a source, and force generation to follow it. One preprocessor, one model, one scale.

**Tracing paper over a sketch.** Lay a sheet of edges over the canvas and the artist paints a brand-new scene but keeps the outlines lined up. ControlNet is that overlay: it hands the model a structure to trace while the prompt decides what actually fills it in.

The gain: ControlNet = a structure overlay on a frozen model. You give it a map (edges/depth/pose), not a photo.

**📝 `controlnet.py`** - *diffusers*

In [ ]:
# pip install diffusers controlnet_aux transformers accelerate torch
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
from controlnet_aux import CannyDetector

# STEP 1: preprocess the source into a CONDITION MAP (not the raw photo!)
canny = CannyDetector()
source = load_image("building.png")
edges = canny(source, low_threshold=100, high_threshold=200)  # edge map; low/high = edge sensitivity (100/200 are common defaults)

# STEP 2: load the ControlNet branch + the base pipeline
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet, torch_dtype=torch.float16).to("cuda")

# STEP 3: generate - the prompt fills in content, the edges fix structure
image = pipe(
    prompt="a neon cyberpunk tower at night, highly detailed",
    image=edges,
    controlnet_conditioning_scale=0.7,  # 0=ignore edges, 1=trace rigidly
    num_inference_steps=28,
).images[0]
image.save("controlled.png")
print("same edges, new content")
# Output: same edges, new content

- **The preprocessor comes first** - `CannyDetector` turns the source photo into an edge map. You pass that *map* as `image=edges`, never the raw photo. Forgetting this (or passing an already-processed map through a second preprocessor) is the classic beginner bug.

- **ControlNet is a branch, not a new model** - `ControlNetModel` loads alongside the frozen SDXL base and injects the condition into it. Swap the model + preprocessor to change condition type (depth, pose, scribble).

- **`controlnet_conditioning_scale`** is the adherence dial: ~0.7 follows the structure with room for content, near 1.0 traces it rigidly (stiff, artifact-prone), near 0 ignores it. *Flux (a newer base-model family, like SDXL) ControlNets want a lower scale (~0.4-0.6) than SDXL (~0.7-1.0)*.

- `control_guidance_start` / `control_guidance_end` (not shown) time *when* the ControlNet is active during the denoising steps - apply it early for layout, release it late for detail (early denoising steps set the coarse composition and late steps add fine detail, from Lesson 6.1).

- **Setup carried from 6.2** - `torch_dtype=torch.float16` loads the model in half precision to halve GPU memory, and `.to("cuda")` puts it on the GPU (use `.to("cpu")` if you have no CUDA GPU, though it is far slower).

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  ControlNet conditions the frozen denoiser on a **structure map**: a preprocessor extracts edges/depth/pose from a source, a ControlNet branch injects it, and `controlnet_conditioning_scale` sets how strictly the output follows it. The prompt still supplies content; the map fixes composition. Different condition types (Canny, depth, OpenPose, scribble) control different aspects of structure.

---

## 🎯 Concept 3: IP-Adapter - An Image as a Prompt

### IP-Adapter - An Image as a Prompt

Show the model a reference picture and it transfers the style, palette, or subject - with zero training.

**Showing the barista a photo of the latte art you want.** You do not train them - you just hold up a picture and say "like this". They keep making *your* drink but match the look. IP-Adapter holds up a reference image to the model as a second, visual prompt.

The gain: IP-Adapter = an image used as a prompt, no training. Text says the content; the reference says the look.

**📝 `ip_adapter.py`** - *diffusers*

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
from diffusers.utils import load_image

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# load the IP-Adapter once; it adds image cross-attention to the frozen model
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                     weight_name="ip-adapter_sdxl.bin")

reference = load_image("style_reference.jpg")   # the LOOK we want
pipe.set_ip_adapter_scale(0.6)                    # 0=ignore image, 1=copy it hard

image = pipe(
    prompt="a mountain palace at golden hour",       # the CONTENT we want
    ip_adapter_image=reference,                        # the reference as a visual prompt
    num_inference_steps=28,
).images[0]
image.save("styled.png")
print("content from text, look from the reference")
# Output: content from text, look from the reference

- `load_ip_adapter` adds image-conditioning cross-attention to the frozen pipeline - like ControlNet, it is an add-on, not a retrain. Load it once, reuse it.

- `ip_adapter_image` is your reference; the model encodes it with CLIP and treats those features as a **visual prompt** alongside the text. Zero training, any reference.

- `set_ip_adapter_scale` is the strength dial: ~0.6 blends the reference's look with your prompt; near 1.0 copies the reference hard (and can override the prompt); near 0 ignores it.

- Variants specialize: **FaceID** (an IP-Adapter variant that locks a specific person's face, not a general style) for identity, **plus/style** for stronger style transfer. IP-Adapters now exist for SDXL, SD3, and Flux.

> ℹ️ **Info**
>
> 🎡 IP-Adapter vs a LoRA vs img2img - three ways to borrow a look
> - **IP-Adapter** - a *reference image* at inference, zero training. Best for "match this one picture's style/subject" on the fly.
> - **LoRA** (from 6.2) - trained on ~15-30 images, a small adapter file. Best for a *reusable* brand style or a specific character you will use again and again.
> - **img2img** (from 6.2) - starts from the reference itself and reworks it. Best for *editing that exact image*, not making new scenes in its style.

#### 💡 What just happened

⚡ What just happened?
  IP-Adapter turns a **reference image into a prompt**: it encodes the image with CLIP and injects those features through extra cross-attention, so the frozen model transfers the reference's style or subject with no training. `set_ip_adapter_scale` dials how strongly. Text supplies content, the reference supplies the look - and they stack with ControlNet (structure + style at once).

---

## 🎯 Concept 4: Composing Control - Multi-ControlNet and Regional Prompting

### Composing Control - Multi-ControlNet and Regional Prompting

Stack conditions, use one union model for many, and give each region of the image its own prompt.

**One recipe card for a two-dish plate.** Write "spicy curry and sweet rice pudding" on a single card and the flavors run together. Give each half of the plate its own card and they stay separate. Regional prompting hands each area of the image its own prompt card.

The gain: a prompt per region stops attributes bleeding across the image. And multiple ControlNets each pin a different aspect.

**📝 `compose_control.py`** - *diffusers*

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel

# MULTI-CONTROLNET: pass a LIST of ControlNets, each with its own scale.
# Here: a pose skeleton fixes the body, a depth map fixes the 3D layout.
pose_cn  = ControlNetModel.from_pretrained("thibaud/controlnet-openpose-sdxl-1.0",
                                           torch_dtype=torch.float16)
depth_cn = ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0",
                                           torch_dtype=torch.float16)

# Each ControlNet still needs its OWN preprocessed map (exactly like Step 2)
# - never the raw photo. Build one condition map per ControlNet:
from controlnet_aux import OpenposeDetector, MidasDetector
from diffusers.utils import load_image

source    = load_image("dancer.png")
pose_map  = OpenposeDetector.from_pretrained("lllyasviel/Annotators")(source)
depth_map = MidasDetector.from_pretrained("lllyasviel/Annotators")(source)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=[pose_cn, depth_cn], torch_dtype=torch.float16).to("cuda")

image = pipe(
    prompt="a dancer on a temple stage, dramatic light",
    image=[pose_map, depth_map],                       # one map per ControlNet
    controlnet_conditioning_scale=[0.8, 0.5],       # pose tight, depth loose
    num_inference_steps=28,
).images[0]
print("pose + depth composed")
# Output: pose + depth composed

# UNION MODEL: one ControlNet that accepts many condition types
# (InstantX FLUX.1-dev-Controlnet-Union). Convenient, but weaker at any one
# condition type than a specialized single-condition ControlNet - a tradeoff.

- **Multi-ControlNet** - pass a *list* of ControlNets and a matching list of condition maps and scales. Each pins a different aspect: here pose (tight, 0.8) and depth (loose, 0.5).

- **Per-control scales matter** - give the control you care most about the higher scale and loosen the rest, or they fight and muddy the result.

- **Union models** (InstantX ControlNet-Union for Flux, SDXL union checkpoints) put many condition types in *one* model - convenient, but generally weaker at any single condition type than a specialized ControlNet. A clear convenience-vs-quality tradeoff.

- **Regional prompting** binds a different prompt to each region of the canvas, so multi-subject scenes stop bleeding attributes (the animation below shows it). There is no first-party `diffusers` call for it yet: use a community regional pipeline (the "regional prompter" / attention-couple pipelines) or a ComfyUI regional-conditioning node from 6.2, passing a mask plus a prompt per region.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  Controls **compose**. **Multi-ControlNet** stacks conditions (pose + depth), each with its own scale. A **union model** packs many condition types into one model - convenient but weaker per mode. **Regional prompting** binds a prompt to a region so multi-subject scenes stop bleeding colours. The skill is spending a limited control budget where it matters.

---

## 🎯 Concept 5: Choosing and Tuning Control

### Choosing and Tuning Control

Structure, style, subject, or region - each has a right tool. And each model has its own scale.

**A toolbox, not a hammer.** A stuck screw, a loose nail, and a bent pipe each need a different tool. "Get the pose right", "match this style", "keep two subjects apart" are different jobs too - reaching for ControlNet for all of them is like hammering a screw.

The gain: match the control tool to the kind of control you need. And tune its scale per model.

| You want to control... | Reach for | Why |
|---|---|---|
| Where things are(pose, layout, edges, depth) | ControlNet | Conditions on a structure map; exact spatial control. |
| The lookfrom one reference image | IP-Adapter | Reference as a visual prompt; zero training. |
| A reusable brand style / character | LoRA (6.2) | Trained adapter; consistent across many generations. |
| Editing one existing image | img2img / inpaint (6.2) | Reworks the input itself, in place. |
| Different content per area | Regional prompting | Binds a prompt to a region; stops attribute bleed. |

**📝 `choose_control.py`** - *decision*

In [ ]:
# Match the tool to the axis of control, then start from a per-model scale.
def pick_control(goal: str) -> str:
    return {
        "pose or layout":          "ControlNet (structure)",
        "match one reference":       "IP-Adapter",
        "reusable brand style":      "LoRA (Lesson 6.2)",
        "edit this image":           "img2img / inpaint (Lesson 6.2)",
        "different content per area": "regional prompting",
    }[goal]

# conditioning_scale starting points differ by model family
SCALE = {"sdxl": 0.75, "flux": 0.5, "sd3.5": 0.5}

print(pick_control("pose or layout"), "| flux scale:", SCALE["flux"])
# Output: ControlNet (structure) | flux scale: 0.5

- `pick_control` is just the toolbox as a lookup: name the axis (where / look / reusable style / edit / region) and it names the tool. The real skill is diagnosing the axis, not memorizing APIs.

- `SCALE` encodes the per-model gotcha: a Stable Diffusion XL conditioning scale is too strong on Flux or SD3.5. Start from these, then tune.

- Two failure signatures to keep in mind: control ignored -> the condition map is wrong; stiff traced look -> the scale is too high.

#### 💡 What just happened

⚡ What just happened?
  Controlled generation is a **toolbox**, and choosing well is most of the skill: ControlNet for structure, IP-Adapter for a reference look, a LoRA for a reusable style, img2img/inpaint for editing, regional prompting for per-area content. Then tune the scales *per model* - SDXL and Flux want different conditioning strengths - and read the two classic failure signatures (ignored control = bad map; traced look = scale too high).

---

## 🎯 Concept 6: Control in Production

### Control in Production

Reproducible control pipelines, the cost of extra models, and the consent line you must not cross.

**A film set needs release forms, not just cameras.** Better gear (more controls) does not remove the need for permission to use someone's likeness. In production, the paperwork gates the shoot.

The gain: production control = reproducible pipeline + cost awareness + a consent gate. The pixels are the easy part.

> 📦 **Info**
>
> 🛠 Taking control to production
> - **Reproducibility** - a control pipeline is preprocessor + control models + scales + seed. Pin all of them (a ComfyUI graph from 6.2, or a versioned script) so a result is repeatable, not a lucky seed.
> - **Cost** - every ControlNet and IP-Adapter adds VRAM (the GPU's video memory) and time on top of the base (a Flux base plus a ControlNet needs roughly 16 GB VRAM at minimum, so two controls can push a 24 GB card to its limit). Use the fewest controls that do the job; deeper cost work is Modules 12-13.
> - **Safety and consent** - keep the output safety checker from 6.2, and add a likeness/consent gate for pose and FaceID control. Recreating a real person without permission is the line; responsible generation is Module 15.
> - **Finishing** - control fixes composition; a finishing pass (face restoration, upscaling) polishes detail. Named here, not the focus.

**📝 `production_run.py`** - *production*

In [ ]:
import json

# A control run is fully described by these fields - pin ALL of them so a
# result is reproducible, not a lucky seed.
run = {
    "base": "stabilityai/stable-diffusion-xl-base-1.0",
    "controlnets": [("openpose", 0.8), ("depth", 0.5)],
    "ip_adapter": {"ref": "brand_style.jpg", "scale": 0.6},
    "seed": 42, "steps": 28,
}

def consent_gate(run, has_consent: bool) -> bool:
    """Block any likeness (pose / FaceID) without recorded consent."""
    # conservative: flag ANY IP-Adapter ref (it may be of a person) or a pose map;
    # in real use, narrow the IP-Adapter check to FaceID / person references.
    uses_identity = bool(run["ip_adapter"]) or \
        any("pose" in c for c, _ in run["controlnets"])
    return (not uses_identity) or has_consent

print("ship" if consent_gate(run, has_consent=True) else "blocked")
# Output: ship
print(json.dumps(run["controlnets"]))
# Output: [["openpose", 0.8], ["depth", 0.5]]

- The `run` dict is the whole recipe - base, ControlNets + scales, IP-Adapter + scale, seed, steps. Pin every field (a versioned script, or a ComfyUI graph from Lesson 6.2) and the result is repeatable, not seed-luck.

- `consent_gate` makes the ethical rule executable: if the run uses identity control - a pose skeleton, or (conservatively) any IP-Adapter reference, since it could be of a real person - it ships only with recorded consent. It runs *before* generation, alongside the output safety checker from 6.2.

- Every ControlNet and IP-Adapter here is extra VRAM and time on the base model - keep the list to what the composition needs. Deep cost/deployment is Modules 12-13; the consent rule deepens in Module 15.

#### 💡 What just happened

⚡ What just happened?
  Shipping control means making it **reproducible** (pin preprocessor, models, scales, seed), **affordable** (each control costs VRAM and time - use the fewest that work), and **responsible** (a consent/likeness gate on pose and FaceID control, plus the safety checker from 6.2). Control gives you precision; production gives you precision you can trust and repeat.

## Synthesis: one brief through the whole control stack

### The complete picture, in one breath

Take the brief "our mascot, in a hero pose, in the brand's watercolor look, for a banner with a clear text area". **Structure:** an OpenPose **ControlNet** pins the hero pose (scale tuned per model). **Look:** an **IP-Adapter** reference (or a brand LoRA from 6.2) carries the watercolor style. **Region:** **regional prompting** keeps the banner's text area clean while the mascot fills the rest. **Compose** them with per-control scales so they cooperate instead of fighting, then **ship** it reproducibly - pinned models, scales, and seed - with a consent gate on the likeness. A prompt gave you content; control gave you the exact composition you were hired to deliver.

> ℹ️ **Info**
>
> Where this goes next in Module 6
> - **Lesson 6.4 - ViT & CLIP:** the CLIP image encoder that powers IP-Adapter - how text and images share one embedding space.
> - **Lesson 6.6 - Video & 3D Generation:** control extended across time (consistent pose and motion between frames).
> - **Modules 12-13 and 15:** deployment and cost of multi-model control pipelines, and the consent/likeness and responsible-generation rules named here.

- Zhang, Rao & Agrawala, *Adding Conditional Control to Text-to-Image Diffusion Models* (ControlNet, 2023) - [arxiv.org/abs/2302.05543](https://arxiv.org/abs/2302.05543)

- Ye et al., *IP-Adapter: Text Compatible Image Prompt Adapter* (2023) - [arxiv.org/abs/2308.06721](https://arxiv.org/abs/2308.06721)

- Mou et al., *T2I-Adapter* (2023) - [arxiv.org/abs/2302.08453](https://arxiv.org/abs/2302.08453)

- Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models* (2022) - [arxiv.org/abs/2112.10752](https://arxiv.org/abs/2112.10752)

- Hugging Face *diffusers* - ControlNet and IP-Adapter guides - [huggingface.co/docs/diffusers](https://huggingface.co/docs/diffusers/using-diffusers/controlnet)

- InstantX FLUX.1-dev-ControlNet-Union - [huggingface.co/InstantX/FLUX.1-dev-Controlnet-Union](https://huggingface.co/InstantX/FLUX.1-dev-Controlnet-Union) · controlnet_aux preprocessors - [github.com/huggingface/controlnet_aux](https://github.com/huggingface/controlnet_aux)

## Recap

> ✅ **Info**
>
> What we covered
> - **The control problem** - text controls content, not composition; layout reshuffles across seeds.
> - **ControlNet** - a branch on the frozen denoiser conditioned on a structure map (edges/depth/pose); `conditioning_scale` is the adherence dial.
> - **IP-Adapter** - a reference image as a visual prompt via CLIP features and cross-attention; zero training.
> - **Composing control** - multi-ControlNet, union models, and regional prompting; control is a budget.
> - **Choosing and tuning** - match the tool to structure/style/subject/region; tune scales per model.
> - **Production** - reproducible pipelines, cost of extra models, and a consent gate on likeness.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Controlled Generation- Place Exactly What You Want, Where You Want It**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.3.html` - regenerate this notebook whenever the source HTML is updated.*
